# Run Full Pipeline

Operator interface for the complete pipeline. Run cells from top to bottom, or resume from any stage whose inputs already exist.

The notebook exposes intermediate reports and artifacts after every stage. Google credentials are required only for Drive-backed scan and document extraction.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "pipeline.json").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "pipeline.json").exists():
    raise FileNotFoundError("Open this notebook from inside the Nursind repository")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from IPython.display import display
from notebooks import interface as nb
from notebooks.shared_config import load_notebook_context

ctx = load_notebook_context(repo_root / "pipeline.json")
paths = ctx.paths
step_cfg = ctx.step("full_pipeline")
display(nb.pipeline_overview(ctx, "full_pipeline"))


## Run Controls

Set a stage to `False` when resuming from existing artifacts.


In [ ]:
VERBOSE = True
RUN_SCAN = True
RUN_DOCUMENTS = True
RUN_EVENTS = True
RUN_FILTER = True
RUN_PAIRING = True
RUN_ENRICHMENT = True
RUN_SUMMARY = True

{
    "scan": RUN_SCAN,
    "documents": RUN_DOCUMENTS,
    "events": RUN_EVENTS,
    "filter": RUN_FILTER,
    "pairing": RUN_PAIRING,
    "enrichment": RUN_ENRICHMENT,
    "summary": RUN_SUMMARY,
}


## Current Artifact State


In [ ]:
display(nb.artifact_table({
    "scan index": paths.scan_included_index,
    "document report": paths.documents_report,
    "events": paths.events_csv,
    "cleaned events": paths.cleaned_events_csv,
    "pairing report": paths.pairing_report,
    "enrichment report": paths.enrichment_report,
    "summary": paths.summary_csv,
}))


## 1. Scan Drive


In [ ]:
if RUN_SCAN:
    from core.drive.auth_service import load_creds
    from core.drive.drive_client import get_drive_service
    from core.drive.logging_utils import setup_logging
    from core.drive.scan.runtime import run_scan

    setup_logging(VERBOSE)
    scan_cfg = ctx.step("scan")
    creds = load_creds()
    drive = get_drive_service(creds)
    scan_report = run_scan(
        creds=creds,
        drive=drive,
        root_id=ctx.root_id,
        workers=int(scan_cfg.get("workers", 8)),
        included_path=str(paths.scan_included_index),
        filtered_path=str(paths.scan_filtered_index),
        report_path=str(paths.scan_report),
    )
    display(nb.report_summary(scan_report))
else:
    scan_report = nb.preview_json(paths.scan_report)
    display(nb.report_summary(scan_report))


## 2. Extract Documents


In [ ]:
if RUN_DOCUMENTS:
    from core.documents.options import ExtractDocumentsFromIndexOptions
    from core.documents.runtime import run_extraction

    document_cfg = ctx.step("extract_documents")
    document_options = ExtractDocumentsFromIndexOptions(
        out=str(paths.documents_dir),
        index=str(paths.scan_included_index),
        included=str(paths.documents_included_index),
        excluded=str(paths.documents_excluded_index),
        report=str(paths.documents_report),
        workers=int(document_cfg.get("workers", 8)),
        download_workers=document_cfg.get("download_workers"),
        extract_workers=int(document_cfg.get("extract_workers", 1)),
        max_in_flight=int(document_cfg.get("max_in_flight", 128)),
        flush_every=int(document_cfg.get("flush_every", 100)),
        limit=int(document_cfg.get("limit", 0)),
        log_every=int(document_cfg.get("log_every", 50)),
        min_normal_score=float(document_cfg.get("min_normal_score", 0.72)),
        min_score_delta=float(document_cfg.get("min_score_delta", 0.08)),
        verbose=VERBOSE,
    )
    document_exit_code = run_extraction(document_options, configure_logging=False)
document_report = nb.preview_json(paths.documents_report)
display(nb.report_summary(document_report))


## 3. Extract Events


In [ ]:
if RUN_EVENTS:
    from core.events.extraction.options import ExtractEventsFromTextOptions
    from core.events.extraction.service import run_from_options as run_extract_events

    event_cfg = ctx.step("extract_events")
    event_report = run_extract_events(ExtractEventsFromTextOptions(
        input_dir=str(paths.documents_dir),
        output_dir=str(paths.events_dir),
        report_json=str(paths.events_report),
        max_pattern_examples=int(event_cfg.get("max_pattern_examples", 12)),
        max_unmatched_examples_per_file=int(event_cfg.get("max_unmatched_examples_per_file", 5)),
        verbose=VERBOSE,
    ))
else:
    event_report = nb.preview_json(paths.events_report)
display(nb.report_summary(event_report))
display(nb.preview_csv(paths.events_csv))


## 4. Filter Midnight Events


In [ ]:
if RUN_FILTER:
    from core.events.filtering.options import FilterMidnightEventsOptions
    from core.events.filtering.service import run_from_options as run_filter_events

    filter_cfg = ctx.step("filter_midnight")
    filter_report = run_filter_events(FilterMidnightEventsOptions(
        input_dir=str(paths.events_dir),
        report_json=str(paths.filter_report),
        removed_csv=str(paths.removed_midnight_csv),
        max_removed_examples_per_file=int(filter_cfg.get("max_removed_examples_per_file", 10)),
        verbose=VERBOSE,
    ))
else:
    filter_report = nb.preview_json(paths.filter_report)
display(nb.report_summary(filter_report))
display(nb.preview_csv(paths.cleaned_events_csv))


## 5. Pair Events


In [ ]:
if RUN_PAIRING:
    from core.shifts.pairing.options import PairEmployeeEventsOptions
    from core.shifts.pairing.runtime import run_from_options as run_pair_events

    pair_cfg = ctx.step("pair_events")
    pair_report = run_pair_events(PairEmployeeEventsOptions(
        input_dir=str(paths.events_dir),
        output_dir=str(paths.shifts_dir),
        report_json=str(paths.pairing_report),
        max_gap_hours=float(pair_cfg.get("max_gap_hours", 16.0)),
        keep_inferred_column=bool(pair_cfg.get("keep_inferred_column", False)),
        verbose=VERBOSE,
    ))
else:
    pair_report = nb.preview_json(paths.pairing_report)
display(nb.report_summary(pair_report))
display(nb.file_table(paths.shifts_dir, "*.pairs.csv"))


## 6. Enrich Shifts


In [ ]:
if RUN_ENRICHMENT:
    from core.shifts.enrichment.options import TurniEnrichmentOptions
    from core.shifts.enrichment.service import run_from_options as run_enrich_shifts

    enrich_cfg = ctx.step("enrich_shifts")
    enrichment_report = run_enrich_shifts(TurniEnrichmentOptions(
        input_dir=str(paths.shifts_dir),
        output_dir=str(paths.enrichment_dir),
        min_hours=float(enrich_cfg.get("min_hours", 6.0)),
        include_holidays=bool(enrich_cfg.get("include_holidays", True)),
        report_json=str(paths.enrichment_report),
        verbose=VERBOSE,
    ))
else:
    enrichment_report = nb.preview_json(paths.enrichment_report)
display(nb.report_summary(enrichment_report))
display(nb.file_table(paths.enrichment_dir, "*.enriched.csv"))


## 7. Summarize Shifts


In [ ]:
if RUN_SUMMARY:
    from core.shifts.summary.options import TurniEmployeeSummaryOptions
    from core.shifts.summary.service import run_from_options as run_summary

    summary_cfg = ctx.step("summarize_shifts")
    output_format = str(summary_cfg.get("format", "csv"))
    summary_path = paths.summary_csv if output_format == "csv" else paths.summary_csv.with_suffix(".json")
    summary_report = run_summary(TurniEmployeeSummaryOptions(
        enriched_dir=str(paths.enrichment_dir),
        out=str(summary_path),
        report_json=str(paths.summary_report),
        year_start=int(summary_cfg.get("year_start", 2014)),
        year_end=int(summary_cfg.get("year_end", 2025)),
        output_format=output_format,
        min_hours=summary_cfg.get("min_hours"),
        verbose=VERBOSE,
    ))
else:
    summary_cfg = ctx.step("summarize_shifts")
    output_format = str(summary_cfg.get("format", "csv"))
    summary_path = paths.summary_csv if output_format == "csv" else paths.summary_csv.with_suffix(".json")
    summary_report = nb.preview_json(paths.summary_report)
display(nb.report_summary(summary_report))
if output_format == "csv":
    display(nb.preview_csv(summary_path, rows=20))
else:
    display(nb.preview_json(summary_path))


## Final Artifact State


In [ ]:
display(nb.artifact_table({
    "scan index": paths.scan_included_index,
    "document report": paths.documents_report,
    "events": paths.events_csv,
    "cleaned events": paths.cleaned_events_csv,
    "pairing report": paths.pairing_report,
    "enrichment report": paths.enrichment_report,
    "summary": summary_path,
    "summary report": paths.summary_report,
}))
